<a href="https://colab.research.google.com/github/MayerT1/LiDAR_Dev/blob/main/Target_GEDI_Export_Sewanee_Pixel_FUSION_GEDI_RS_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install earthaccess harmony-py hvplot folium geopandas geoviews

In [2]:
from harmony import BBox, Client, Collection, Request, CapabilitiesRequest
import h5py
from datetime import datetime
import json
import earthaccess
import geopandas as gp
import pandas as pd
import numpy as np
import os
from IPython.display import JSON
from shapely.geometry import Point
import hvplot.pandas
import folium
from folium import GeoJson
from IPython.display import display, HTML
import requests
from io import StringIO
from datetime import datetime
import seaborn as sns
from matplotlib import cm, colors
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap
import textwrap
from mpl_toolkits.mplot3d import Axes3D
# gv.extension('bokeh', 'matplotlib')
from holoviews import opts
import shapely
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as colors

os.chdir('../../')

In [34]:
%cd /content

/content


In [35]:
!mkdir Module_2_GEDI_L2b

!mkdir Module_2_GEDI_L2b/Sewanee_Domain

In [36]:
%cd /content/Module_2_GEDI_L2b

/content/Module_2_GEDI_L2b


In [3]:
auth = earthaccess.login(persist=True)
# auth.token

Enter your Earthdata Login username: tjm0042
Enter your Earthdata password: ··········


In [4]:
harmony_client = Client(auth=(auth.username, auth.password))

In [5]:
# Bring in aois
geojson_urls = {
    "Sewanee_Domain": "https://raw.githubusercontent.com/SERVIR/GEDI_Earthhub_ARSET_Training/main/AOIs/Sewanee_Domain.geojson",
}

In [6]:
# Load GeoJSONs into GeoDataFrames and store map HTML
geojson_gdfs = {}
maps_html = []

def create_map_and_gdf(name, url):
    # Get GeoJSON from URL
    geojson_data = requests.get(url).json()

    # Create GeoDataFrame
    gdf = gp.GeoDataFrame.from_features(geojson_data["features"])
    geojson_gdfs[name] = gdf

    # Determine map center
    coords = geojson_data['features'][0]['geometry']['coordinates']
    geom_type = geojson_data['features'][0]['geometry']['type']

    if geom_type == "Polygon":
        coords = coords[0]
    elif geom_type == "MultiPolygon":
        coords = coords[0][0]
    else:
        raise ValueError(f"Unsupported geometry type: {geom_type}")

    lat = sum(pt[1] for pt in coords) / len(coords)
    lon = sum(pt[0] for pt in coords) / len(coords)

    # Create map
    fmap = folium.Map(location=[lat, lon], zoom_start=13, tiles="OpenStreetMap")
    folium.GeoJson(geojson_data, name=name).add_to(fmap)

    # Return rendered HTML
    return fmap._repr_html_()

# Generate maps and GeoDataFrames
for name, url in geojson_urls.items():
    html_map = create_map_and_gdf(name, url)
    maps_html.append(html_map)


In [20]:
# # Display maps side by side
# html = f"""
# <div style="display: flex; flex-wrap: wrap;">
#     {''.join([f'<div style="flex: 1; min-width: 400px; margin: 5px;">{m}</div>' for m in maps_html])}
# </div>
# """
# display(HTML(html))

In [7]:
##Pull out the geopanda dataframes
Sewanee_Domain = geojson_gdfs["Sewanee_Domain"]


In [8]:
# Store all bbox objects in a dictionary for later use
bbox_dict = {}

for name, url in geojson_urls.items():
    gdf = gp.read_file(url)
    minx, miny, maxx, maxy = gdf.total_bounds
    bbox = BBox(minx, miny, maxx, maxy)
    bbox_dict[f"{name}_roi"] = bbox

bbox_dict

{'Sewanee_Domain_roi': BBox: West:-85.98971714751701, South:35.127110504508586, East:-85.85727611112807, North:35.235639374867105}

In [9]:
capabilities = harmony_client.submit(CapabilitiesRequest(short_name='GEDI02_B'))
print(json.dumps(capabilities, indent=2))

{
  "conceptId": "C2142776747-LPCLOUD",
  "shortName": "GEDI02_B",
  "variableSubset": true,
  "bboxSubset": true,
  "shapeSubset": true,
  "temporalSubset": true,
  "concatenate": false,
  "reproject": false,
  "outputFormats": [
    "application/x-hdf"
  ],
  "services": [
    {
      "name": "sds/trajectory-subsetter",
      "href": "https://cmr.earthdata.nasa.gov/search/concepts/S2836723123-XYZ_PROV",
      "capabilities": {
        "subsetting": {
          "temporal": true,
          "bbox": true,
          "shape": true,
          "variable": true
        },
        "output_formats": [
          "application/x-hdf"
        ]
      }
    }
  ],
  "variables": [
    {
      "name": "/BEAM0000/algorithmrun_flag",
      "href": "https://cmr.earthdata.nasa.gov/search/concepts/V2837647264-LPCLOUD"
    },
    {
      "name": "/BEAM0000/ancillary/dz",
      "href": "https://cmr.earthdata.nasa.gov/search/concepts/V2837647474-LPCLOUD"
    },
    {
      "name": "/BEAM0000/ancillary/l2a_al

In [10]:
print(capabilities['shortName'], ',', capabilities['conceptId'])

concept_id = capabilities['conceptId']
print("concept_id:", concept_id)

GEDI02_B , C2142776747-LPCLOUD
concept_id: C2142776747-LPCLOUD


In [11]:
append_url = "https://raw.githubusercontent.com/SERVIR/GEDI_Earthhub_ARSET_Training/main/append_field.txt"
append_text = requests.get(append_url).text

start_index = append_text.find("{")
end_index = append_text.rfind("}") + 1
dict_str = append_text[start_index:end_index]

append_dict = eval(dict_str)

In [12]:
# print only the dictionary fo avaible GEDI 2 B products to explore
gedi_l2b = append_dict.get('GEDI_L2B', {})

# Step 4: Print items under 'GEDI_L2B'
print("Items under 'GEDI_L2B':")
for item in gedi_l2b:
    print(item)

Items under 'GEDI_L2B':
rx_processing/rg_eg_flag_a4
rx_processing/rg_eg_gamma_error_a5
rx_processing/rg_eg_gamma_error_a1
rx_processing/rg_error_a3
rx_processing/rg_eg_amplitude_a2
geolocation/local_beam_elevation
ancillary/rg_eg_constraint_center_buffer
rx_processing/rx_energy_a6
ancillary/tx_noise_stddev_multiplier
pgap_theta_z
rx_processing/rg_eg_center_error_a5
rx_processing/pgap_theta_a4
land_cover_data/modis_nonvegetated
geolocation/longitude_bin0_error
rx_processing/rg_error_a1
geolocation/elev_highestreturn
land_cover_data/landsat_water_persistence
land_cover_data/leaf_on_doy
rx_processing/algorithmrun_flag_a5
geolocation/lon_lowestmode
rx_processing/rx_energy_a4
geolocation/digital_elevation_model
rhov
rx_processing/rg_a5
rx_processing/rg_error_a2
geolocation/longitude_lastbin_error
rx_processing/rg_a6
rx_processing/rg_eg_gamma_a1
l2b_quality_flag
rx_processing/rg_eg_amplitude_error_a4
rx_processing/rg_eg_amplitude_a3
rx_processing/pgap_theta_error_a3
land_cover_data/leaf_off_

In [13]:
#Selecting the 'GEDI02_B' of intrest to then be downloaded
subset_L2B = ['geolocation/lat_lowestmode', 'geolocation/lon_lowestmode', 'geolocation/degrade_flag', 'geolocation/digital_elevation_model','geolocation/elev_lowestmode','geolocation/elev_highestreturn','l2b_quality_flag','rh100', 'pai', 'fhd_normal']

In [14]:
### filter those varibales ofintrest out of the larger list
selected_L2B = []
for s in subset_L2B:
    my_var = [v for v in gedi_l2b if v.endswith(f'{s}')]
    if len(my_var) == 1:
        selected_L2B.append(my_var[0])

    elif len(my_var) > 1:
        my_var = [v for v in my_var if v.startswith(f'{s}')]

        for l in my_var:
            if l not in selected_L2B:
                selected_L2B.append(l)

selected_L2B

['geolocation/lat_lowestmode',
 'geolocation/lon_lowestmode',
 'geolocation/degrade_flag',
 'geolocation/digital_elevation_model',
 'geolocation/elev_lowestmode',
 'geolocation/elev_highestreturn',
 'l2b_quality_flag',
 'rh100',
 'pai',
 'fhd_normal']

In [15]:
#Selecting a few beams: ['BEAM0000', 'BEAM0001', 'BEAM0010', 'BEAM0011', 'BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']
beams = ['BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']

In [16]:
#Append our beams of intrest to our variables of intrest
#the subset list will then be used throughout
subset = []
for b in beams:
    beam_subset = [f'/{b}/{layer}' for layer in selected_L2B]
    [subset.append(i) for i in beam_subset]
subset

['/BEAM0101/geolocation/lat_lowestmode',
 '/BEAM0101/geolocation/lon_lowestmode',
 '/BEAM0101/geolocation/degrade_flag',
 '/BEAM0101/geolocation/digital_elevation_model',
 '/BEAM0101/geolocation/elev_lowestmode',
 '/BEAM0101/geolocation/elev_highestreturn',
 '/BEAM0101/l2b_quality_flag',
 '/BEAM0101/rh100',
 '/BEAM0101/pai',
 '/BEAM0101/fhd_normal',
 '/BEAM0110/geolocation/lat_lowestmode',
 '/BEAM0110/geolocation/lon_lowestmode',
 '/BEAM0110/geolocation/degrade_flag',
 '/BEAM0110/geolocation/digital_elevation_model',
 '/BEAM0110/geolocation/elev_lowestmode',
 '/BEAM0110/geolocation/elev_highestreturn',
 '/BEAM0110/l2b_quality_flag',
 '/BEAM0110/rh100',
 '/BEAM0110/pai',
 '/BEAM0110/fhd_normal',
 '/BEAM1000/geolocation/lat_lowestmode',
 '/BEAM1000/geolocation/lon_lowestmode',
 '/BEAM1000/geolocation/degrade_flag',
 '/BEAM1000/geolocation/digital_elevation_model',
 '/BEAM1000/geolocation/elev_lowestmode',
 '/BEAM1000/geolocation/elev_highestreturn',
 '/BEAM1000/l2b_quality_flag',
 '/BEAM

In [22]:
# Define temporal range
temporal_range = {'start': datetime(2019, 4, 1),
                  'stop': datetime(2021, 12, 30)}

In [37]:
# Corresponding output directories
output_dirs = [
    '/content/Module_2_GEDI_L2b/Sewanee_Domain'
]

In [38]:
# Loop through paired spatial ROIs and directories
for (spatial, out_dir) in zip(bbox_dict.keys(), output_dirs):
    print(f"\nSubmitting request for: {out_dir}")

    request = Request(
        collection=Collection(id=concept_id),
        spatial=bbox_dict[spatial],  # Use the corresponding BBox from the dictionary
        temporal=temporal_range,
        variables=subset
    )

    print("Check if the request payload is valid:", request.is_valid())

    task = harmony_client.submit(request)
    print(f'Harmony request ID: {task}')

    print('Processing your Harmony request:')
    task_json = harmony_client.result_json(task, show_progress=True)

    results = harmony_client.download_all(task, directory=out_dir, overwrite=True)
    file_names = [f.result() for f in results]

    print(f"Download completed for {out_dir}")



Submitting request for: /content/Module_2_GEDI_L2b/Sewanee_Domain
Check if the request payload is valid: True
Harmony request ID: 94ef895a-a9e8-4745-bd5b-f59a11632946
Processing your Harmony request:


 [ Processing: 100% ] |###################################################| [|]


/content/Module_2_GEDI_L2b/Sewanee_Domain/102580703_GEDI02_B_2019122150008_O02186_03_T04733_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580704_GEDI02_B_2019138014405_O02426_02_T03841_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580705_GEDI02_B_2019209043943_O03530_03_T03157_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580706_GEDI02_B_2019210205202_O03556_02_T00995_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580708_GEDI02_B_2019312044317_O05128_02_T02571_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580710_GEDI02_B_2020182144232_O08778_03_T00311_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580711_GEDI02_B_2020196021509_O08987_02_T05264_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domain/102580713_GEDI02_B_2021131101046_O13658_03_T04580_02_003_01_V002_subsetted.h5
/content/Module_2_GEDI_L2b/Sewanee_Domai

Partition

In [39]:
# Authenticate and initialize Earth Engine
import ee
import geemap
import geemap.chart as chart
ee.Authenticate()
ee.Initialize(project='servir-sco-assets')
Map = geemap.Map()


In [41]:
#//////////////////////////////////////////////////////////////////////////////////
GEDIindicesA_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/GEDIindicesA_2021")
ROI = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Sewanee_Domain')

glad = ee.Image("projects/glad/GLCLU2020/Forest_height_2020").clip(ROI)
Map.addLayer(glad, {}, "glad")
glad = glad.gte(1).select(['b1']).rename(['class']);
#print("glad", glad)

classMask = GEDIindicesA_2021.addBands(glad)
Map.addLayer(classMask, {}, "classMask")
#print("classMask", classMask)


proj = GEDIindicesA_2021.projection()
grid = ROI.geometry().coveringGrid(proj, 30)
grid = ee.FeatureCollection(grid).randomColumn("random", 42);
#print("spatial_partition number of boxes in the grid", grid.size())
val_samp = grid.filter('random <= 0.1').set("samp_type","val_samp");
test_samp = grid.filter('random <= 0.3 and random >= 0.1').set("samp_type","test_samp");
train_samp = grid.filter('random >= 0.3').set("samp_type","train_samp");


#rint('val_samp 10%', val_samp.size());
Map.addLayer(val_samp, {"color": "blue"}, "val_samp 10%")

#print('test_samp 20%', test_samp.size());
Map.addLayer(test_samp, {"color": "red"}, "test_samp 20%")

#print('train_samp 70%', train_samp.size());
Map.addLayer(train_samp, {"color": "green"}, "train_samp 70%")

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [44]:
import os
import h5py
import pandas as pd
import geopandas as gp
from shapely.geometry import Point
from datetime import datetime
import ee
from tqdm import tqdm

# Initialize EE if not already
# ee.Initialize()

# Constants
SAMPLE_PERCENT = 0.0001
GEDI_H5_FOLDER = '/content/Module_2_GEDI_L2b/Sewanee_Domain'

# Load GEE Data
GEDIindicesA_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/GEDIindicesA_2021")
ROI = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Sewanee_Domain')

# Grid sampling
proj = GEDIindicesA_2021.projection()
grid = ROI.geometry().coveringGrid(proj, 30)
grid = ee.FeatureCollection(grid).randomColumn("random", 42)

val_samp = grid.filter('random <= 0.1')
test_samp = grid.filter('random > 0.1 and random <= 0.3')
train_samp = grid.filter('random > 0.3')

# Optional downsampling
def sample_fc_randomly(fc, percent=100, seed=42):
    fc = fc.randomColumn("rand", seed)
    return fc.filter(ee.Filter.lt("rand", percent))

train_samp = sample_fc_randomly(train_samp, SAMPLE_PERCENT)
val_samp = sample_fc_randomly(val_samp, SAMPLE_PERCENT)
test_samp = sample_fc_randomly(test_samp, SAMPLE_PERCENT)

# Convert EE FeatureCollection to shapely-compatible GeoDataFrame
def fc_to_gdf(fc):
    geojson = fc.getInfo()
    features = geojson['features']
    gdf = gp.GeoDataFrame.from_features(features)
    gdf.set_crs("EPSG:4326", inplace=True)
    return gdf

train_gdf = fc_to_gdf(train_samp)
val_gdf = fc_to_gdf(val_samp)
test_gdf = fc_to_gdf(test_samp)

# Combine and tag
train_gdf["split"] = "train"
val_gdf["split"] = "val"
test_gdf["split"] = "test"

sample_gdf = pd.concat([train_gdf, val_gdf, test_gdf], ignore_index=True)

# Convert GEE geometries to bounding boxes
sample_gdf["bbox"] = sample_gdf.geometry.bounds.values.tolist()

# H5 → DataFrame
def h5_to_dataframe(h5_path, beams, vars):
    gedi_ds = h5py.File(h5_path, 'r')
    product = gedi_ds['METADATA']['DatasetIdentification'].attrs['shortName']
    fileName = gedi_ds['METADATA']['DatasetIdentification'].attrs['fileName']
    date = datetime.strptime(fileName.rsplit('_')[2], '%Y%j%H%M%S').strftime('%Y-%m-%d %H:%M:%S')

    df_beam = pd.DataFrame(columns=vars)

    for b in beams:
        data_dic = {}
        try:
            for v in vars:
                value = gedi_ds[f'{b}/{v}'][()]
                data_dic[v] = value.tolist()
            beam_df = pd.DataFrame(data_dic)
            beam_df.insert(0, 'product', product)
            beam_df.insert(1, 'Beam', b)
            beam_df.insert(2, 'fileName', fileName)
            beam_df.insert(3, 'date', date)
            df_beam = pd.concat([df_beam, beam_df], ignore_index=True)
        except Exception as e:
            print(f"⚠️ Skipping beam {b} in file {fileName}: {e}")
            continue

    return df_beam.reset_index(drop=True)

# Spatial join GEDI with sample sites
def spatial_join_gedi(gedi_df):
    if 'lat_lowestmode' in gedi_df.columns and 'lon_lowestmode' in gedi_df.columns:
        gedi_df = gedi_df.rename(columns={
            'geolocation/lat_lowestmode': 'lat',
            'geolocation/lon_lowestmode': 'lon'
        })
    gedi_df = gp.GeoDataFrame(
        gedi_df,
        geometry=gedi_df.apply(lambda row: Point(row.lon, row.lat), axis=1),
        crs="EPSG:4326"
    )
    joined = gp.sjoin(gedi_df, sample_gdf, how="inner", predicate='within')
    return joined

# Process all h5 files
all_h5_files = [os.path.join(GEDI_H5_FOLDER, f) for f in os.listdir(GEDI_H5_FOLDER) if f.endswith('.h5')]
gedi_final = []

print(f"Processing {len(all_h5_files)} GEDI files...")
for h5_file in tqdm(all_h5_files):
    try:
        df = h5_to_dataframe(h5_file, beams, subset_L2B)
        if not df.empty:
            joined = spatial_join_gedi(df)
            if not joined.empty:
                gedi_final.append(joined)
    except Exception as e:
        print(f"⚠️ Error in file {h5_file}: {e}")
        continue

# Final concatenated GeoDataFrame
if gedi_final:
    gedi_all = pd.concat(gedi_final, ignore_index=True)
    print(f"✅ Final GEDI sample size: {gedi_all.shape}")
else:
    gedi_all = pd.DataFrame()
    print("⚠️ No valid GEDI data found.")


Processing 15 GEDI files...


 20%|██        | 3/15 [00:00<00:00, 28.87it/s]

⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580720_GEDI02_B_2021298160332_O16250_03_T10425_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Skipping beam BEAM1011 in file GEDI02_B_2019312044317_O05128_02_T02571_02_003_01_V002.h5: 'Unable to synchronously open object (component not found)'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580708_GEDI02_B_2019312044317_O05128_02_T02571_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580704_GEDI02_B_2019138014405_O02426_02_T03841_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Skipping beam BEAM1011 in file GEDI02_B_2021232110553_O15224_02_T11109_02_003_01_V002.h5: 'Unable to synchronously open object (component not found)'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580717_GEDI02_B_2021232110553_O15224_02_T11109_02_003_01_V002_subsetted.h5: 'Series' object has no at

 47%|████▋     | 7/15 [00:00<00:00, 33.35it/s]

⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580710_GEDI02_B_2020182144232_O08778_03_T00311_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'


 73%|███████▎  | 11/15 [00:00<00:00, 29.74it/s]

⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580706_GEDI02_B_2019210205202_O03556_02_T00995_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580719_GEDI02_B_2021288125533_O16093_02_T09533_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580711_GEDI02_B_2020196021509_O08987_02_T05264_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580715_GEDI02_B_2021139070445_O13780_03_T06003_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580709_GEDI02_B_2019337015939_O05514_03_T01887_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ Skipping beam BEAM0101 in file GEDI02_B_2021238154634_O15320_03_T07579_02_003_01_V002.h5: 'Unable to synchronous

100%|██████████| 15/15 [00:00<00:00, 30.69it/s]

⚠️ Error in file /content/Module_2_GEDI_L2b/Sewanee_Domain/102580705_GEDI02_B_2019209043943_O03530_03_T03157_02_003_01_V002_subsetted.h5: 'Series' object has no attribute 'lon'
⚠️ No valid GEDI data found.


In [46]:
gedi_final

[]

In [48]:
#save the dataframes as csvs
for out_dir in output_dirs:
    # Extract folder name only (e.g., 'SPB_AOI')
    aoi_name = os.path.basename(out_dir)

    # Get corresponding GeoDataFrame
    gdf = geo_dfs.get(aoi_name)

    if gdf is not None and not gdf.empty:
        # Define CSV path
        csv_path = os.path.join(out_dir, f"{aoi_name}_GEDI_L2B.csv")

        # Drop the geometry column (optional, if you only want flat CSV)
        gdf_no_geom = gdf.drop(columns='geometry')

        # Save as CSV
        gdf_no_geom.to_csv(csv_path, index=False)
        print(f"✅ Saved: {csv_path}")
    else:
        print(f"⚠️ Skipped: {aoi_name} (no data)")


AttributeError: 'list' object has no attribute 'get'